# 21 — The $T_1$ and $T_2$ Channels of a Real Qubit

`01/20` derived the general evolution of an open qubit — a channel, written with Kraus operators — and built dephasing, amplitude damping and depolarizing, each at a fixed strength. What it left out is time. In `01/16` a qubit idled before readout, and its coherence fell as $e^{-t/T_2}$ while its $\langle Z\rangle$ rose toward $1$ (the qubit relaxing to $|0\rangle$) as $1 - e^{-t/T_1}$; today you put two of `01/20`'s channels on a clock and find that those two laws are exactly what they become. The $T_1$ and $T_2$ are those of a stored calibration snapshot of a real device, and the idle is simulated from them.

You will derive by hand why a memoryless decay must be exponential, compose amplitude damping with dephasing and prove that the order does not matter, derive $1/T_2 = 1/2T_1 + 1/T_\varphi$, and rebuild from the calibration snapshot the channel behind the idle data of `01/16`, checked against the one Aer builds by its own route. Finally you will see that $T_2 \le 2T_1$ is the condition for the idle map to be a channel: a map that broke it would not even send states to states. That closes Module 01.

**Prerequisites:** `01/20_quantum_channels` (Kraus operators, the test on half a Bell pair, dephasing and amplitude damping and their Bloch maps), `01/16_tomography_of_a_noisy_qubit` (the idle sweep, the calibration sheet of qubit $3$, the meter factor $1 - 2\epsilon$), `01/13_from_states_to_density_matrix` (the Bloch vector from the entries of $\rho$), `01/14_pure_vs_mixed` (purity).

**What you'll be able to do**
- Show that composing two amplitude dampings multiplies their survival probabilities, and derive from that why the decay is $e^{-t/T_1}$.
- Prove entry by entry that amplitude damping and dephasing commute, compose them, and derive $1/T_2 = 1/2T_1 + 1/T_\varphi$.
- Read $T_1$ and $T_2$ off a calibration snapshot, compute $T_\varphi$, and reproduce the idle laws of `01/16` — saying which part of the agreement is algebra and which part is a test.
- Show with the half-Bell-pair test of `01/20` that $T_2 \le 2T_1$ is the condition for the idle map to be a channel, and that the boundary is reached.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit.quantum_info import DensityMatrix, Kraus
from qiskit_aer.noise import thermal_relaxation_error

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.quantum.states import KET_1, KET_PLUS
from qot_course.quantum.gates import PAULI_X
from qot_course.quantum.density import density_matrix, purity, bloch_vector, density_from_bloch
from qot_course.quantum.composite import bell_state, apply_channel, is_cptp

rng = np.random.default_rng(0)   # random qubit states for the composition and commutation checks
viz.use_course_style()


def sheet_T1_T2(phys, snapshot="FakeManilaV2"):
    """T1 and T2 (seconds) of one physical qubit, read offline from a stored IBM calibration snapshot."""
    from qiskit_ibm_runtime import fake_provider
    q = getattr(fake_provider, snapshot)().target.qubit_properties[phys]
    return q.t1, q.t2


def sheet_readout_error(phys, snapshot="FakeManilaV2"):
    """Calibrated readout error of one physical qubit from the same snapshot (the epsilon of 01/16)."""
    from qiskit_ibm_runtime import fake_provider
    return getattr(fake_provider, snapshot)().target["measure"][(phys,)].error


# the two channels of 01/20 section 4 (the course library has no function for them)
def dephasing(lmbda):
    return [np.diag([1, np.sqrt(1 - lmbda)]).astype(complex), np.diag([0, np.sqrt(lmbda)]).astype(complex)]

def amplitude_damping(gamma):
    return [np.diag([1, np.sqrt(1 - gamma)]).astype(complex), np.sqrt(gamma) * np.array([[0, 1], [0, 0]], dtype=complex)]

def random_states(n):
    # n density matrices with Bloch vectors drawn uniformly in the ball
    vs = rng.normal(size=(n, 3))
    return [density_from_bloch(v * rng.uniform() ** (1 / 3) / np.linalg.norm(v)) for v in vs]

## 1. Amplitude damping on a clock: why the decay is exponential

Start with energy relaxation. In `01/20` §4, amplitude damping of strength $\gamma$ had Kraus operators $K_0 = \mathrm{diag}(1, \sqrt{1-\gamma})$ and $K_1 = \sqrt\gamma\,|0\rangle\langle1|$: the excited population survives with probability $1 - \gamma$ ($\rho_{11} \to (1-\gamma)\rho_{11}$), the lost part lands in $|0\rangle$, and the coherence shrinks by the square root, $\rho_{01} \to \sqrt{1-\gamma}\,\rho_{01}$. Now let the qubit idle. Which $\gamma$ belongs to an idle of length $t$?

Compose two idles by hand. Apply strength $\gamma_1$, then strength $\gamma_2$: $\rho_{11} \to (1-\gamma_2)(1-\gamma_1)\,\rho_{11}$, $\rho_{01} \to \sqrt{1-\gamma_2}\sqrt{1-\gamma_1}\,\rho_{01}$, and $\rho_{00}$ gains whatever $\rho_{11}$ lost, because the trace is kept. That is again an amplitude damping, with
$$
1 - \gamma = (1-\gamma_1)(1-\gamma_2).
$$
Survival probabilities multiply: the excitation has to survive the first idle and then the second.

Now add one physical assumption: the environment keeps no memory that acts back on the qubit. In the recipe of `01/20` §1, each slice of idle meets a fresh environment in $|0\rangle_E$; the records written by earlier slices are carried away and never return. It is the Markov property of the chain in `01/20` §3: the next step depends only on the present. Then an idle of length $t + u$ is an idle of length $t$ followed by an idle of length $u$, whatever happened before. Write $s(t) = 1 - \gamma(t)$ for the survival after time $t$. The composition rule becomes $s(t+u) = s(t)\,s(u)$, with $s(0) = 1$. A family of maps that composes like this, with the idle of length zero doing nothing, is called a **semigroup**; the prefix records that the maps cannot be run backward. Logarithms turn the product into a sum ($s$ never reaches $0$: $s(t) = s(t/2)^2$, and continuity at $s(0) = 1$ rules out a zero), $\ln s(t+u) = \ln s(t) + \ln s(u)$, and the only continuous functions that add like that are straight lines through the origin. So $\ln s(t) = -t/T_1$ for one constant $T_1$, and
$$
1 - \gamma(t) = e^{-t/T_1}.
$$
The exponential is the only law that composes, and $T_1$ is the one number left to measure. (The memoryless environment is a model, standard for a qubit weakly coupled to a large environment, and not a theorem.) A straight-line rule such as $\gamma = t/T_1$ agrees with it for short idles but cannot compose: two half-idles would not equal one whole idle, and the line would reach a survival of zero at $t = T_1$ and go negative after.

On the Bloch ball the map of `01/20` §4 becomes a flow in time:
$$
(x,\ y,\ z) \;\longmapsto\; \big(e^{-t/2T_1}x,\ \ e^{-t/2T_1}y,\ \ 1 - e^{-t/T_1}(1 - z)\big).
$$
Every state slides toward the north pole $|0\rangle$. Started from $|1\rangle$ ($z = -1$), the excited population is $e^{-t/T_1}$; started from $|+\rangle$ ($z = 0$), the height is $z(t) = 1 - e^{-t/T_1}$, the $\langle Z\rangle$ law of `01/16`. Let's check the composition rule on random states, cut an idle into many short slices, and watch the straight-line rule fail, with the $T_1$ of the qubit `01/16` idled, taken from the snapshot of `01/12`, read as a calibration sheet in `01/16`.

In [ ]:
PHYS = 3                                                     # the qubit 01/16 idled
T1, T2 = sheet_T1_T2(PHYS)
print(f"qubit {PHYS} of the FakeManilaV2 snapshot: T1 = {T1*1e6:.1f} us, T2 = {T2*1e6:.1f} us")

# (1) composition by hand: strength g2 after strength g1 is strength 1 - (1-g1)(1-g2)
g1, g2 = 0.2, 0.35
g12 = 1 - (1 - g1) * (1 - g2)
gap = max(np.abs(apply_channel(apply_channel(r, amplitude_damping(g1)), amplitude_damping(g2))
                 - apply_channel(r, amplitude_damping(g12))).max() for r in random_states(200))
print(f"AD({g2}) after AD({g1}) vs AD({g12:.2f}) on 200 random states: largest entry difference {gap:.1e}")

# (2) an idle of 150 us cut into n equal slices composes back to one exponential
t_end = 150e-6
for n in [1, 3, 10, 100]:
    rho = density_matrix(KET_1)
    for _ in range(n):
        rho = apply_channel(rho, amplitude_damping(1 - np.exp(-t_end / n / T1)))
    print(f"  {n:3d} slices of {t_end / n * 1e6:6.1f} us: excited population {rho[1, 1].real:.6f}   e^(-t/T1) = {np.exp(-t_end / T1):.6f}")

# (3) the straight-line rule gamma = t/T1 does not compose
print(f"straight-line rule over 150 us: survival {1 - t_end / T1:.4f} in one go, {(1 - t_end / 2 / T1) ** 2:.4f} as two halves")
print(f"at t = T1 the exponential leaves a survival of e^-1 = {np.exp(-1):.3f}; the straight line leaves 0")

t_us = np.linspace(0, 400, 401)
surv = np.exp(-t_us * 1e-6 / T1)
t_slices = np.arange(0, 401, 25)
rho, pts = density_matrix(KET_1), [1.0]
for _ in t_slices[1:]:                                       # 16 slices of 25 us, composed one after the other
    rho = apply_channel(rho, amplitude_damping(1 - np.exp(-25e-6 / T1))); pts.append(rho[1, 1].real)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))
ax = axes[0]
ax.plot(t_us, surv, color=COLORS["highlight"], lw=2.5, label=r"$e^{-t/T_1}$")
ax.plot(t_slices, pts, "o", color=COLORS["highlight"], ms=6, mfc="white", label="25-us slices of amplitude damping, composed")
line = 1 - t_us * 1e-6 / T1
ax.plot(t_us[line >= 0], line[line >= 0], color=COLORS["muted"], lw=1.5, ls="--", label=r"straight line $1 - t/T_1$")
ax.axvline(T1 * 1e6, color=COLORS["grid"], lw=1.2); ax.text(T1 * 1e6 + 5, 0.03, r"$t = T_1$", fontsize=10, color=COLORS["text"])
ax.set_xlabel("idle time (us)"); ax.set_ylabel(r"excited population $\rho_{11}$, from $|1\rangle$")
ax.set_title("Survival multiplies,\nso the decay is an exponential", pad=12)
ax.legend(fontsize=9, loc="upper right")
ax = axes[1]
angles = np.linspace(0, 2 * np.pi, 241)
ax.plot(np.sin(angles), np.cos(angles), color=COLORS["muted"], lw=1.5, ls="--", label=r"$t = 0$: the sphere")
for t_snap, alpha in zip([50, 150, 400], [0.9, 0.7, 0.5]):
    a = np.exp(-t_snap * 1e-6 / (2 * T1))
    ax.plot(a * np.sin(angles), 1 - a ** 2 * (1 - np.cos(angles)), color=COLORS["highlight"], lw=2.2, alpha=alpha, label=fr"$t = {t_snap}\,\mu$s")
ax.plot(np.zeros_like(t_us), 1 - 2 * surv, color=COLORS["text"], lw=1.2, ls=":", label=r"path of $|1\rangle$")
ax.set_aspect("equal"); ax.set_xlim(-1.2, 1.2); ax.set_ylim(-1.2, 1.2)
ax.set_xlabel(r"$x$"); ax.set_ylabel(r"$z$"); ax.set_title("The ball under amplitude damping,\nshrinking toward the north pole", pad=12)
ax.legend(fontsize=8, loc="lower right")
plt.tight_layout(); plt.show()

**Read the output and the figure.** The snapshot gives qubit $3$ the times `01/16` used, $T_1 = 179.1\,\mu$s and $T_2 = 54.4\,\mu$s. The composition rule holds on $200$ random states to rounding error: strength $0.35$ after strength $0.2$ is strength $0.48$, because $0.8 \times 0.65 = 0.52$ survives. Cutting a $150\,\mu$s idle into $1$, $3$, $10$ or $100$ slices gives the same excited population, $0.433$, equal to $e^{-t/T_1}$ to six decimals: the semigroup at work. The straight-line rule fails the same test, with a survival of $0.1625$ in one go and $0.3378$ as two halves.

On the left, the open circles (sixteen slices of $25\,\mu$s, composed) sit on the exponential, while the dashed straight line, its tangent at $t = 0$, hits zero at $t = T_1$, where the exponential still leaves $e^{-1} = 0.368$. On the right, the dashed circle is the sphere of pure states, and the rose curves are its images after $50$, $150$ and $400\,\mu$s: ellipses that shrink sideways at the rate $1/2T_1$ and slide upward until they collapse on the north pole. The dotted line is the path of $|1\rangle$, straight up the $z$ axis. The qubit of `01/16` lost its coherence far faster than this, and section 2 finds why.

## 2. Dephasing joins in: one $T_2$ from two channels

Dephasing goes on a clock by the same argument. Its strength enters only through the coherence factor $\sqrt{1-\lambda}$, two idles multiply their factors, and a memoryless environment forces $\sqrt{1 - \lambda(t)} = e^{-t/T_\varphi}$, with a **pure dephasing time** $T_\varphi$. (Equivalently $\lambda(t) = 1 - e^{-2t/T_\varphi}$, the line the code uses.)

A real qubit undergoes both at once. Does the order matter? Write what each channel does to the three numbers that fix a qubit state, $\rho_{00}$, $\rho_{11}$ and $\rho_{01}$ (with $\rho_{10} = \rho_{01}^*$):

- amplitude damping: $(\rho_{00},\ \rho_{11},\ \rho_{01}) \to (\rho_{00} + \gamma\rho_{11},\ \ (1-\gamma)\rho_{11},\ \ \sqrt{1-\gamma}\,\rho_{01})$;
- dephasing: $(\rho_{00},\ \rho_{11},\ \rho_{01}) \to (\rho_{00},\ \ \rho_{11},\ \ \sqrt{1-\lambda}\,\rho_{01})$.

Whichever comes first, the populations see only the amplitude damping, because dephasing leaves them alone, and the coherence is multiplied by $\sqrt{1-\gamma}$ and by $\sqrt{1-\lambda}$, in one order or the other. The two products are the same numbers, so the two compositions agree on every input: these two channels commute. Because they commute at every strength, cutting the idle into many short slices that alternate the two — the qubit undergoing both at once — gives the same channel as one full amplitude damping followed by one full dephasing. That is a property of this pair and not of channels in general; a bit flip and amplitude damping, for example, do not commute (exercise 1).

So the coherence picks up *both* factors: $\rho_{01}(t) = \rho_{01}(0)\,e^{-t/2T_1}\,e^{-t/T_\varphi}$. The measured $T_2$ is defined by $\rho_{01}(t) \propto e^{-t/T_2}$, so

$$
\frac{1}{T_2} = \frac{1}{2T_1} + \frac{1}{T_\varphi} \qquad\Longrightarrow\qquad T_2 \le 2T_1 ,
$$

with equality only when there is no pure dephasing at all. That is the law `01/16` left as a promise: energy relaxation *alone* already costs coherence at half its own rate, and dephasing only adds to it.

Now the calibration snapshot. It lists $T_1$ and $T_2$ but not $T_\varphi$, which the formula supplies: $T_\varphi = 1/(1/T_2 - 1/2T_1)$. Be clear about one thing before the comparison with `01/16`. Because $T_\varphi$ is computed from $T_2$, the composed channel reproduces $e^{-t/T_2}$ by construction, so that match checks the algebra above and tests nothing. Two checks do test something. The first swaps the order of the two channels on random states, which the proof above says cannot matter. The second compares the composition with a channel somebody else built from the same two numbers: the thermal-relaxation channel of Aer (Aer's name for the idle noise it builds from $T_1$ and $T_2$), the simulator behind the idle data of `01/16`. Let's run the idle with the composed channel and lay it over the `01/16` laws.

In [ ]:
T_phi = 1 / (1 / T2 - 1 / (2 * T1))                          # pure dephasing time implied by the sheet
print(f"qubit {PHYS}: T1 = {T1*1e6:.1f} us, T2 = {T2*1e6:.1f} us  ->  2 T1 = {2*T1*1e6:.1f} us >= T2: {T2 <= 2*T1},   T_phi = {T_phi*1e6:.1f} us")
print(f"share of the coherence decay rate 1/T2 that comes from energy relaxation: T2 / 2T1 = {T2 / (2 * T1):.2f}")

def idle_channel(t):
    # amplitude damping for time t and pure dephasing for time t (the order does not matter: proved above, checked below)
    gamma = 1 - np.exp(-t / T1)
    lam = 1 - np.exp(-2 * t / T_phi)                         # sqrt(1 - lam) = e^{-t/T_phi}
    return amplitude_damping(gamma), dephasing(lam)

rho_plus = density_matrix(KET_PLUS)
t_us = np.linspace(0, 150, 151)
X_out, Z_out, pur = [], [], []
for t in t_us * 1e-6:
    kA, kD = idle_channel(t)
    rho = apply_channel(apply_channel(rho_plus, kA), kD)
    r = bloch_vector(rho)
    X_out.append(r[0]); Z_out.append(r[2]); pur.append(purity(rho))
X_out, Z_out, pur = map(np.array, (X_out, Z_out, pur))
x_law = np.exp(-t_us * 1e-6 / T2)                            # the 01/16 laws (meter factor 1 here: no readout error)
z_law = 1 - np.exp(-t_us * 1e-6 / T1)
print("channel <X>(t) == e^{-t/T2}:", np.allclose(X_out, x_law), "  (built in: T_phi came from T2)",
      "   channel <Z>(t) == 1 - e^{-t/T1}:", np.allclose(Z_out, z_law))
print(f"minimum purity along the idle: {pur.min():.3f} at t = {t_us[pur.argmin()]:.0f} us")
eps = sheet_readout_error(PHYS)
r2_dip = 2 * pur.min() - 1                                   # |r|^2 at the dip
print(f"through the meter of 01/16 (epsilon = {eps:.4f}, factor 1 - 2 epsilon = {1 - 2 * eps:.4f}): purity at the dip reads {(1 + (1 - 2 * eps) ** 2 * r2_dip) / 2:.3f}")

# two checks that are not built in: (1) the two orders, on random states and on |+>;
# (2) Aer's thermal-relaxation channel, built from the same T1, T2 by its own route
gap_order, gap_aer = 0.0, 0.0
probes = [rho_plus] + random_states(20)
for t in t_us * 1e-6:
    kA, kD = idle_channel(t)
    aer_kraus = Kraus(thermal_relaxation_error(T1, T2, t).to_quantumchannel())
    for r0 in probes:
        ad_then_dp = apply_channel(apply_channel(r0, kA), kD)
        dp_then_ad = apply_channel(apply_channel(r0, kD), kA)
        aer = DensityMatrix(r0).evolve(aer_kraus).data
        gap_order = max(gap_order, np.abs(ad_then_dp - dp_then_ad).max())
        gap_aer = max(gap_aer, np.abs(ad_then_dp - aer).max())
print(f"largest entry difference over {len(t_us)} idle times x {len(probes)} input states: order swapped {gap_order:.1e},  vs Aer's thermal relaxation {gap_aer:.1e}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
ax = axes[0]
ax.plot(t_us, X_out, color=COLORS["quantum"], lw=2.5, label=r"$\langle X\rangle$ from the channel (coherence)")
ax.plot(t_us[::10], x_law[::10], "o", color=COLORS["quantum"], ms=6, mfc="white", label=r"$e^{-t/T_2}$  (01/16 law)")
ax.plot(t_us, Z_out, color=COLORS["highlight"], lw=2.5, label=r"$\langle Z\rangle$ from the channel (relaxation)")
ax.plot(t_us[::10], z_law[::10], "s", color=COLORS["highlight"], ms=6, mfc="white", label=r"$1 - e^{-t/T_1}$  (01/16 law)")
ax.plot(t_us, np.exp(-t_us * 1e-6 / (2 * T1)), color=COLORS["muted"], lw=1.2, ls=":", label=r"$e^{-t/2T_1}$: coherence from $T_1$ alone")
ax.set_xlabel("idle time (us)"); ax.set_ylabel("Pauli average"); ax.set_title("Amplitude damping + dephasing\nreproduce the 01/16 idle laws", pad=12)
ax.legend(fontsize=8, loc="upper right")
ax = axes[1]
ax.plot(X_out, Z_out, color=COLORS["flow"], lw=2.5)
ax.scatter([X_out[0]], [Z_out[0]], color=COLORS["source"], zorder=5, label=r"$t = 0$: $|+\rangle$")
ax.scatter([X_out[-1]], [Z_out[-1]], color=COLORS["target"], zorder=5, label=r"$t = 150\,\mu$s: toward $|0\rangle$")
circle = plt.Circle((0, 0), 1, fill=False, color=COLORS["grid"], lw=1.5)
ax.add_patch(circle); ax.set_aspect("equal"); ax.set_xlim(-1.1, 1.1); ax.set_ylim(-1.1, 1.1)
ax.set_xlabel(r"$\langle X\rangle$"); ax.set_ylabel(r"$\langle Z\rangle$"); ax.set_title("The Bloch vector's path in the x–z plane:\ninto the ball, then up toward the pole", pad=12)
ax.legend(fontsize=8, loc="lower left")
plt.tight_layout(); plt.show()

**Read the output and the figure.** The calibration sheet of qubit $3$ says $T_2 \approx 54\,\mu$s against $2T_1 \approx 358\,\mu$s — comfortably inside the law, with a pure dephasing time $T_\varphi \approx 64\,\mu$s doing most of the damage: energy relaxation accounts for only $0.15$ of the coherence decay rate. Composing amplitude damping (with $\gamma = 1 - e^{-t/T_1}$) and dephasing (with $\sqrt{1-\lambda} = e^{-t/T_\varphi}$) lands on the two laws that `01/16` laid over its tomography data — `allclose` on both. As announced, the $\langle X\rangle$ match is the algebra of the formula above and not an independent test; the $\langle Z\rangle$ match is the semigroup of section 1.

The independent checks are on the last printed line. Swapping the order of the two channels changes no entry beyond rounding error ($2.2\times10^{-16}$ at most), on $|+\rangle$ and on twenty random states, at every one of the $151$ idle times, as the proof says. And the composition agrees, to rounding error ($1.3\times10^{-15}$ at most), with the thermal-relaxation channel that Aer builds from the same $T_1$ and $T_2$ by its own route: the channel that produced the idle data of `01/16`, whose tomography followed these laws within its error bars at nearly every point (51 of 54 residuals within two error bars).

In the figure, the dotted line on the left shows what coherence would do under $T_1$ alone ($e^{-t/2T_1}$, slow); the gap to the solid blue line is $T_\varphi$ at work. On the right, the Bloch vector leaves the sphere along the $x$ axis (coherence dies first), dips to a purity of $0.59$ at $73\,\mu$s (the point of the green path nearest the centre; purity is $\tfrac12(1 + |\vec r|^2)$, `01/14`), and climbs toward the north pole $|0\rangle$ as the qubit relaxes. Seen through the meter of `01/16` (the factor $1 - 2\epsilon$, with $\epsilon = 0.0144$ for this qubit), the same dip reads $0.585$, the shallow minimum `01/16` found in its reconstructed purities. The idle sweep of `01/16` came from a simulator that runs exactly these two named channels, built from a real device's stored calibration; you can now write their Kraus operators from memory.

## 3. $T_2 \le 2T_1$: the condition for a channel

Section 2 derived $T_2 \le 2T_1$ inside a model: amplitude damping plus dephasing. Could some other noise give a qubit with $T_2 > 2T_1$? Take the map that has exactly these two behaviours — the populations relax as in amplitude damping, and the coherence is multiplied by some factor $c$, which in time is $e^{-t/T_2}$:
$$
\Lambda_{\gamma,c}:\quad (\rho_{00},\ \rho_{11},\ \rho_{01}) \;\longmapsto\; (\rho_{00} + \gamma\rho_{11},\ \ (1-\gamma)\rho_{11},\ \ c\,\rho_{01}).
$$
It is linear and keeps the trace for every $c$. Is it a channel? Use the test of `01/20` §2: apply it to half a Bell pair. Because $\Lambda$ is linear and B is left alone, $\Lambda\otimes\mathrm{id}$ sends each term $|a\rangle\langle a'|\otimes|b\rangle\langle b'|$ to $\Lambda(|a\rangle\langle a'|)\otimes|b\rangle\langle b'|$, so you need $\Lambda$ on the four **matrix units** $|a\rangle\langle a'|$: $\Lambda(|0\rangle\langle0|) = |0\rangle\langle0|$, $\Lambda(|1\rangle\langle1|) = \gamma|0\rangle\langle0| + (1-\gamma)|1\rangle\langle1|$, $\Lambda(|0\rangle\langle1|) = c\,|0\rangle\langle1|$ and $\Lambda(|1\rangle\langle0|) = c\,|1\rangle\langle0|$. Read $|00\rangle\langle11|$ as $|0\rangle\langle1|\otimes|0\rangle\langle1|$. The Bell state is $\tfrac12\big(|00\rangle\langle00| + |00\rangle\langle11| + |11\rangle\langle00| + |11\rangle\langle11|\big)$, so by hand
$$
(\Lambda\otimes\mathrm{id})(\rho_{\mathrm{Bell}}) = \tfrac12\Big(|00\rangle\langle00| + c\,|00\rangle\langle11| + c\,|11\rangle\langle00| + (1-\gamma)\,|11\rangle\langle11| + \gamma\,|01\rangle\langle01|\Big).
$$
The weight $\gamma/2$ on $|01\rangle$ is harmless. The only part that can fail is the $2\times2$ block on $|00\rangle$ and $|11\rangle$, $\tfrac12\begin{pmatrix}1 & c\\ c & 1-\gamma\end{pmatrix}$, whose determinant is $\tfrac14\big[(1-\gamma) - c^2\big]$. The output is block-diagonal, so its eigenvalues are those of its blocks; this block has positive trace, so its two eigenvalues are both non-negative exactly when their product, the determinant, is (`01/19` §2 read eigenvalues off trace and determinant the same way). The output is a state exactly when
$$
c^2 \le 1 - \gamma .
$$
In time this reads $e^{-2t/T_2} \le e^{-t/T_1}$ for every $t$, that is $2/T_2 \ge 1/T_1$: **past $T_2 = 2T_1$ the map fails the half-Bell test, so it is not a channel.** Here, unlike the transpose of `01/20` §2, the weaker one-qubit test fails at the same place: a pure state close to the north pole is already pushed outside the Bloch ball (exercise 4), so for this family positive and completely positive draw the same line. And the boundary is reached: when $c^2 \le 1 - \gamma$, the map is amplitude damping followed by dephasing with $\sqrt{1-\lambda} = c/\sqrt{1-\gamma}$, a channel you already own. Every pair with $T_2 \le 2T_1$ is a real channel, and none beyond.

The cell scans $T_2$ across $2T_1$ for the $T_1$ of qubit $3$ and an idle of $50\,\mu$s, runs both tests, places the five qubits of the snapshot against the line, and asks Aer to build a channel beyond it.

In [ ]:
t_test = 50e-6
gamma_t = 1 - np.exp(-t_test / T1)

def relax_and_scale(gamma, c):
    # the map Lambda_{gamma, c}, entry by entry (a Kraus list exists only when c^2 <= 1 - gamma)
    def act(m):
        return np.array([[m[0, 0] + gamma * m[1, 1], c * m[0, 1]],
                         [c * m[1, 0], (1 - gamma) * m[1, 1]]], dtype=complex)
    return act

def on_half_bell(act):
    # (Lambda (x) id) on the Bell state: act on the A-block of every B matrix element
    rho4 = density_matrix(bell_state()).reshape(2, 2, 2, 2)          # indices (a, b, a', b')
    out = np.zeros_like(rho4)
    for b in range(2):
        for b_ in range(2):
            out[:, b, :, b_] = act(rho4[:, b, :, b_])
    return out.reshape(4, 4)

angles = np.linspace(0, 2 * np.pi, 20001)
sphere = np.stack([np.sin(angles), np.zeros_like(angles), np.cos(angles)], axis=1)   # pure states in the x-z plane

T2_grid = np.arange(20, 601) * 1e-6
bell_min, one_margin = [], []
for T2_try in T2_grid:
    c = np.exp(-t_test / T2_try)
    bell_min.append(np.linalg.eigvalsh(on_half_bell(relax_and_scale(gamma_t, c))).min())
    r_img = np.stack([c * sphere[:, 0], c * sphere[:, 1], gamma_t + (1 - gamma_t) * sphere[:, 2]], axis=1)
    one_margin.append(1 - (r_img ** 2).sum(axis=1).max())     # >= 0 iff every pure state stays in the ball
bell_min, one_margin = np.array(bell_min), np.array(one_margin)
first_bad_bell = T2_grid[np.argmax(bell_min < -1e-12)]
first_bad_one = T2_grid[np.argmax(one_margin < -1e-12)]
print(f"idle t = {t_test*1e6:.0f} us, T1 = {T1*1e6:.1f} us, 2 T1 = {2*T1*1e6:.1f} us")
print(f"half-Bell test first fails at T2 = {first_bad_bell*1e6:.0f} us;  one-qubit positivity first fails at T2 = {first_bad_one*1e6:.0f} us  (grid step 1 us)")
print(f"at T2 = 600 us: lowest eigenvalue on half a Bell pair {bell_min[-1]:+.4f},  one-qubit margin {one_margin[-1]:+.4f}")

# the boundary is reached: inside it, Lambda_{gamma, c} is amplitude damping followed by dephasing
c_in = np.exp(-t_test / T2)
lam_in = 1 - c_in ** 2 / (1 - gamma_t)
same = all(np.allclose(relax_and_scale(gamma_t, c_in)(r), apply_channel(apply_channel(r, amplitude_damping(gamma_t)), dephasing(lam_in)))
           for r in random_states(50))
print(f"for qubit {PHYS}'s own T2: Lambda == AD then dephasing(lambda = {lam_in:.4f}) on 50 random states: {same}")

print("\nthe five qubits of the snapshot:")
sheet = [sheet_T1_T2(q) for q in range(5)]
for q, (t1, t2) in enumerate(sheet):
    print(f"  qubit {q}: T1 = {t1*1e6:6.1f} us, T2 = {t2*1e6:6.1f} us, T2/2T1 = {t2 / (2 * t1):.2f}, T_phi = {1 / (1 / t2 - 1 / (2 * t1)) * 1e6:6.1f} us")
try:
    thermal_relaxation_error(T1, 2.2 * T1, t_test)
    print("Aer built a channel with T2 = 2.2 T1")
except Exception as err:
    print(f"Aer, asked for T2 = 2.2 T1: {type(err).__name__}: {err}")

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
ax = axes[0]
ax.axhline(0, color=COLORS["text"], lw=0.8)
ax.plot(T2_grid * 1e6, bell_min, color=COLORS["negative"], lw=2.5, label="lowest eigenvalue on half a Bell pair")
ax.plot(T2_grid * 1e6, one_margin, color=COLORS["quantum"], lw=2.5, label=r"one qubit: $1 - \max|\vec r\,'|^2$ over pure states")
ax.axvline(2 * T1 * 1e6, color=COLORS["muted"], lw=1.5, ls="--", label=fr"$2T_1 = {2*T1*1e6:.0f}\,\mu$s")
ax.set_xlabel(r"$T_2$ tried (us)"); ax.set_ylabel("test value (negative = not a state)")
ax.set_title(r"Past $T_2 = 2T_1$ the map stops" + "\nsending states to states", pad=12)
ax.legend(fontsize=8, loc="lower left")
ax = axes[1]
t1s = np.array([s[0] for s in sheet]) * 1e6; t2s = np.array([s[1] for s in sheet]) * 1e6
grid_t1 = np.linspace(0, 220, 50)
ax.fill_between(grid_t1, 2 * grid_t1, 500, color=COLORS["negative"], alpha=0.15, lw=0, label=r"forbidden: $T_2 > 2T_1$")
ax.plot(grid_t1, 2 * grid_t1, color=COLORS["negative"], lw=1.5, ls="--")
ax.scatter(t1s, t2s, color=COLORS["source"], s=50, zorder=5, label="the five qubits of the snapshot")
ax.scatter([t1s[PHYS]], [t2s[PHYS]], color=COLORS["target"], s=90, zorder=6, label=f"qubit {PHYS} (01/16)")
for q in range(5):
    ax.text(t1s[q] + 3, t2s[q] + 8, str(q), fontsize=10, color=COLORS["text"])
ax.set_xlim(0, 220); ax.set_ylim(0, 500)
ax.set_xlabel(r"$T_1$ (us)"); ax.set_ylabel(r"$T_2$ (us)"); ax.set_title("Every qubit of the snapshot\nsits below the line", pad=12)
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout(); plt.show()

**Read the output and the figure.** On the left, the two tests run side by side as $T_2$ is pushed past $2T_1 = 358\,\mu$s for an idle of $50\,\mu$s. Up to the dashed line both stay at zero: the map sends every state to a state (the zero is the empty corner $|10\rangle$ of the Bell output, and the pure states that stay pure, such as $|0\rangle$). Past the line both turn negative, and on the $1\,\mu$s grid both first fail at $359\,\mu$s, the first grid point beyond $2T_1$. The two curves measure different things, so their sizes are not comparable. What they share is the place where they cross zero (by hand, both cross at exactly $c^2 = 1 - \gamma$, that is $T_2 = 2T_1$, so the shared $359\,\mu$s is not an effect of the grid): a qubit with $T_2 > 2T_1$ would turn some ordinary state into something that is not a state. The printed `True` shows the other side of the boundary: with qubit $3$'s own $T_2$, the map is amplitude damping followed by dephasing with $\lambda = 0.7899$, a channel built from Kraus operators.

On the right, the five qubits of the snapshot all sit below the dashed line $T_2 = 2T_1$, from qubit $0$, the closest at $T_2/2T_1 = 0.39$, to qubit $2$, the most dominated by pure dephasing at $0.08$. Aer agrees with the law in its own words: asked for a thermal-relaxation channel with $T_2 = 2.2\,T_1$, it refuses with "T_2 greater than 2 * T_1". The law that `01/16` used and section 2 derived is the condition for a relaxing qubit's idle map to send states to states at all, and no qubit whose idle follows these two exponential laws can break it, on any chip.

## Your turn

1. **Survival multiplies, and order sometimes matters.** (a) Section 1 composed two amplitude dampings through their entries. Redo it with Kraus operators: write the four products $K^{(2)}_j K^{(1)}_i$, show that one vanishes, and check by hand that the other three act on $\rho_{11}$ and $\rho_{01}$ as a single amplitude damping of strength $\gamma_1 + \gamma_2 - \gamma_1\gamma_2$: two Kraus sets, one channel, as in `01/20` exercise 1. (b) Show that the straight-line rule $\gamma(t) = t/T_1$ agrees with $1 - e^{-t/T_1}$ to first order in $t$, and find the idle time at which the two differ by $0.01$ for qubit $3$. (c) Apply the bit flip $\{\sqrt{1-p}\,I, \sqrt p\,X\}$ of `01/20` (exercise 3) and amplitude damping to $|0\rangle$ in both orders, by hand: show that one order leaves $\rho_{11} = p(1-\gamma)$ and the other $\rho_{11} = p$, so these two channels do not commute. Check it in code.
2. **The purity dip, by hand.** From $|+\rangle$ the idle gives $x(t) = e^{-t/T_2}$, $z(t) = 1 - e^{-t/T_1}$ and purity $P(t) = \tfrac12\big(1 + x^2 + z^2\big)$ (`01/14`). Write $dP/dt$, solve $dP/dt = 0$ numerically for qubit $3$, and compare with the dip printed in section 2. Explain in one sentence why the purity has to come back up, whatever $T_1$ and $T_2$ are.
3. **Five qubits, five stories.** For each qubit of the snapshot, compute the idle time after which the coherence of $|+\rangle$ has halved ($T_2\ln2$) and the time after which an excited qubit has a one-half chance of having relaxed ($T_1\ln2$). Which qubit keeps a superposition longest, and which keeps an excitation longest? Rebuild the two-panel figure of section 2 for qubit $2$, and explain from $T_2/2T_1 = 0.08$ why its Bloch path hugs the $z$ axis sooner.
4. **Harder: positivity is already enough.** For the map $\Lambda_{\gamma,c}$ of section 3, show from the entries that its Bloch map is $(x, y, z) \to (cx,\ cy,\ \gamma + (1-\gamma)z)$. Take the pure state $x = \sin\alpha$, $y = 0$, $z = \cos\alpha$ and expand $|\vec r\,'|^2$ for small $\alpha$ to show $|\vec r\,'|^2 = 1 + \big(c^2 - (1-\gamma)\big)\alpha^2 + O(\alpha^3)$. Conclude that when $c^2 > 1-\gamma$ a state near $|0\rangle$ leaves the ball, so the map is not even positive, and $T_2 \le 2T_1$ holds for every map of this form, completely positive or not. Then, for $c^2 \le 1-\gamma$, write the Kraus operators of amplitude damping followed by dephasing with $\sqrt{1-\lambda} = c/\sqrt{1-\gamma}$ as the products $K^{\mathrm D}_j K^{\mathrm A}_i$, show that one product vanishes, and check `is_cptp` on the three that remain.

## What you built

- You can show that composing two amplitude dampings multiplies their survival probabilities, and derive from a memoryless environment that the survival after an idle of length $t$ must be $e^{-t/T_1}$, the only law that composes.
- You can prove entry by entry that amplitude damping and dephasing commute, compose them into one idle channel, and derive $1/T_2 = 1/2T_1 + 1/T_\varphi$ and with it $T_2 \le 2T_1$.
- You can read $T_1$ and $T_2$ off a calibration snapshot, compute $T_\varphi$, rebuild the channel behind the idle sweep of `01/16`, and say what was tested: the order swap and the comparison with Aer's thermal-relaxation channel are independent checks, while the $\langle X\rangle$ match is algebra.
- You can show with the half-Bell-pair test of `01/20` that $T_2 \le 2T_1$ is the condition for the idle map to be a channel, see that the one-qubit test draws the same line here, and show that every pair inside the bound is a real channel.

**Module 01 complete.** Across this module you went from two amplitudes to the Bloch sphere, measurement, gates, expectation values and shot noise; to the density matrix, mixedness, distances and tomography on a noisy device; to two systems, the partial trace, CNOT, entanglement, channels and now the clocks of a real qubit — the full language of quantum states, each piece earned by hand. Before module 02, open [the course dictionary](../dictionary.ipynb) (`notebooks/dictionary.ipynb`) and read its first table top to bottom once: the left column is the probability theory you knew, the right column is what each piece became.

## What's next

With the quantum objects in hand, module `02_InformationAndGeometry` builds the information theory and geometry that classical and quantum optimal transport both stand on: Shannon entropy (`02/01` — where $H(A) \le H(AB)$ and the binary entropy $h(p)$ of `01/19` are proved), relative entropy, mutual information, the probability simplex and its two geometries, then their quantum versions — Umegaki relative entropy, quantum mutual information (`02/11`, where the coins and the Bell pair reappear), negative conditional entropy (`02/12`) and the Bures distance (`02/13`, with the fidelity convention of `01/15`). It opens with `02/01_shannon_entropy`.

## References

- M. A. Nielsen & I. L. Chuang, *Quantum Computation and Quantum Information*, Cambridge University Press (2010) — amplitude damping, phase damping, $T_1$ and $T_2$. doi:10.1017/CBO9780511976667
- P. Krantz, M. Kjaergaard, F. Yan, T. P. Orlando, S. Gustavsson & W. D. Oliver, *A quantum engineer's guide to superconducting qubits*, Appl. Phys. Rev. **6**, 021318 (2019) — $T_1$, $T_2$, $T_\varphi$ and $1/T_2 = 1/2T_1 + 1/T_\varphi$ on real devices. doi:10.1063/1.5089550
- M.-D. Choi, *Completely positive linear maps on complex matrices*, Linear Algebra Appl. **10**, 285 (1975) — complete positivity, the test behind section 3. doi:10.1016/0024-3795(75)90075-0

**Previous:** `notebooks/01_QuantumFoundations/20_quantum_channels.ipynb` · **Next:** `notebooks/02_InformationAndGeometry/01_shannon_entropy.ipynb`